# Merge the LoRA adapter and evaluate on the test split
Run on a **fresh Colab T4 server**, top to bottom. Needs a Hugging Face token (Read access is enough) to download the private adapter repo.

Plan: merge adapter into fp16 base, sanity-check 5 examples, run the full test set, save results to the Hub (Colab's disk is wiped on disconnect).

## 1. Setup

In [8]:
!nvidia-smi -L
!git clone https://github.com/hannna24/Github-IssueHawk.git
%cd Github-IssueHawk/training
!pip install -q -U transformers peft accelerate python-dotenv scikit-learn
# Colab ships an old torchao that newer peft rejects; it is not needed here.
!pip uninstall -y -q torchao


GPU 0: Tesla T4 (UUID: GPU-9cd0bb3b-1bc4-45ef-5661-6a74b8df3429)
Cloning into 'Github-IssueHawk'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 140 (delta 55), reused 123 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 1.55 MiB | 19.33 MiB/s, done.
Resolving deltas: 100% (55/55), done.
/content/Github-IssueHawk/training/Github-IssueHawk/training


## 2. Log in to Hugging Face
Paste your token when prompted (it is not stored in this notebook).

In [2]:
from getpass import getpass
from huggingface_hub import login
login(token=getpass('Hugging Face token: '))

In [7]:
%cd /content/Github-IssueHawk/training
!ls -la ../checkpoints/merged


/content/Github-IssueHawk/training
ls: cannot access '../checkpoints/merged': No such file or directory


## 3. Merge the adapter into the base model
Downloads Qwen2.5-3B-Instruct in fp16 (~6 GB), attaches `final-adapter/` from your Hub repo, merges, and saves to `../checkpoints/merged`. A few minutes.

In [9]:
%cd /content/Github-IssueHawk/training
!python merge_adapter.py --adapter hanabakeer24/issuehawk-qwen2.5-3b-lora --subfolder final-adapter --out ../checkpoints/merged

/content/Github-IssueHawk/training
Loading weights: 100% 434/434 [00:13<00:00, 33.24it/s]

final-adapter/adapter_model.safetensors: downloading bytes:  85% 25.2M/29.5M [00:05<00:00, 16.2MB/s, 1.91MB/s  ]
final-adapter/adapter_model.safetensors: downloading bytes: 100% 27.6M/27.6M [00:05<00:00, 4.98MB/s, 2.29MB/s  ]
final-adapter/adapter_model.safetensors: reconstructing file: 100% 29.5M/29.5M [00:05<00:00, 5.33MB/s, 2.59MB/s  ]
Writing model shards: 100% 1/1 [03:03<00:00, 183.48s/it]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 1.61MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 10.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 15.1MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 18.0MB/s]
merged model saved to /content/Github-IssueHawk/checkpoints/merged


Check that the merge finished: expect `config.json`, several `model-0000x-of-0000y.safetensors` files and the tokenizer files. If this says *No such file*, the merge above failed -- fix that before going on.

In [10]:
!ls -la ../checkpoints/merged

total 6038460
drwxr-xr-x 2 root root       4096 Oct  1 09:09 .
drwxr-xr-x 3 root root       4096 Oct  1 09:06 ..
-rw-r--r-- 1 root root       2507 Oct  1 09:09 chat_template.jinja
-rw-r--r-- 1 root root       1549 Oct  1 09:06 config.json
-rw-r--r-- 1 root root        243 Oct  1 09:06 generation_config.json
-rw------- 1 root root 6171926680 Oct  1 09:09 model.safetensors
-rw-r--r-- 1 root root        694 Oct  1 09:09 tokenizer_config.json
-rw-r--r-- 1 root root   11421892 Oct  1 09:09 tokenizer.json


## 4. Sanity check (5 issues)
Expect `status counts: {'ok': 5}` and raw outputs like `{"label": "bug"}`. If you see extra text or `malformed_json`, stop here. Writes no result files.

In [11]:
%cd /content/Github-IssueHawk/training
!python predict_student.py --model ../checkpoints/merged --limit 5

/content/Github-IssueHawk/training
Loading weights: 100% 434/434 [00:20<00:00, 20.69it/s]
  5/5
status counts: {'ok': 5}
  #60672 true='bug' raw='{"label": "bug"}'
  #60303 true='bug' raw='{"label": "bug"}'
  #64028 true='documentation' raw='{"label": "documentation"}'
  #61224 true='feature-request' raw='{"label": "feature-request"}'
  #59956 true='feature-request' raw='{"label": "feature-request"}'
sanity run only (--limit): not writing result files


## 5. Full test-set evaluation
Runs all test issues, prints coverage, accuracy and macro-F1, and writes `eval/results/predictions_student.jsonl` and `latest_student.json`.

In [12]:
%cd /content/Github-IssueHawk/training
!python predict_student.py --model ../checkpoints/merged

/content/Github-IssueHawk/training
Loading weights: 100% 434/434 [00:23<00:00, 18.37it/s]
  8/352
  16/352
  24/352
  32/352
  40/352
  48/352
  56/352
  64/352
  72/352
  80/352
  88/352
  96/352
  104/352
  112/352
  120/352
  128/352
  136/352
  144/352
  152/352
  160/352
  168/352
  176/352
  184/352
  192/352
  200/352
  208/352
  216/352
  224/352
  232/352
  240/352
  248/352
  256/352
  264/352
  272/352
  280/352
  288/352
  296/352
  304/352
  312/352
  320/352
  328/352
  336/352
  344/352
  352/352
status counts: {'ok': 352}
  #60672 true='bug' raw='{"label": "bug"}'
  #60303 true='bug' raw='{"label": "bug"}'
  #64028 true='documentation' raw='{"label": "documentation"}'
  #61224 true='feature-request' raw='{"label": "feature-request"}'
  #59956 true='feature-request' raw='{"label": "feature-request"}'
{
  "ground_truth_rows": 352,
  "predictions_received": 352,
  "scored": 352,
  "failed": 0,
  "unmatched_issue_number": 0,
  "coverage_rate": 1.0,
  "status_counts": {
    

## 6. Per-class results

In [13]:
import json
r = json.load(open('../eval/results/latest_student.json'))
for k in r['labels']:
    v = r['report'][k]
    print(f"{k:16s} precision {v['precision']:.3f}  recall {v['recall']:.3f}  f1 {v['f1-score']:.3f}  support {int(v['support'])}")
print('accuracy', round(r['report']['accuracy'], 4), ' macro F1', round(r['report']['macro avg']['f1-score'], 4), ' n', r['n'])
print('confusion matrix (rows=true, cols=pred):', r['labels'])
for row in r['confusion_matrix']:
    print(row)

bug              precision 0.957  recall 0.991  f1 0.974  support 227
feature-request  precision 0.908  recall 1.000  f1 0.952  support 59
documentation    precision 0.981  recall 0.773  f1 0.864  support 66
accuracy 0.9517  macro F1 0.93  n 352
confusion matrix (rows=true, cols=pred): ['bug', 'feature-request', 'documentation']
[225, 1, 1]
[0, 59, 0]
[10, 5, 51]


## 7. Save the results to the Hub *(Colab's disk is wiped on disconnect)*
Uploads the two result files to an `eval/` folder in your adapter repo.

In [14]:
from huggingface_hub import HfApi
api = HfApi()
for name in ('predictions_student.jsonl', 'latest_student.json'):
    api.upload_file(
        path_or_fileobj=f'../eval/results/{name}',
        path_in_repo=f'eval/{name}',
        repo_id='hanabakeer24/issuehawk-qwen2.5-3b-lora',
    )
print('uploaded')

uploaded
